<a href="https://colab.research.google.com/github/Divyesh-Developer-Web/EXPERIMENT-BASED-/blob/main/AI_Powered_Career_%26_Cyber_Defence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio fpdf2 requests -q
print("Dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.6/301.6 kB 10.5 MB/s eta 0:00:00
Dependencies installed successfully!


In [ ]:
# PASTE YOUR GEMINI API KEY DIRECTLY HERE
API_KEY = 'PLACE YOU REAL API KEY'  # Replace with your actual key

if API_KEY and len(API_KEY) > 20 and API_KEY.startswith('AIzaSy'):
    print(f"API Key loaded successfully: {API_KEY[:10]}...{API_KEY[-4:]}")
else:
    print("Invalid API key! Please paste your actual Gemini API key.")
    print("Get your key from: https://aistudio.google.com/app/apikey")

Invalid API key! Please paste your actual Gemini API key.
Get your key from: https://aistudio.google.com/app/apikey


In [ ]:
import gradio as gr
import requests
from fpdf import FPDF, XPos, YPos
from datetime import datetime
import os
import re
import time
import uuid
from threading import Lock

if 'API_KEY' not in globals():
    raise RuntimeError("Run CELL 2 first to set your API key!")

print(f"✅ Using API Key: {API_KEY[:10]}...{API_KEY[-4:]}")

GEMINI_LOCK = Lock()
LAST_CALL_TIME = 0
MIN_INTERVAL = 10

class ModernResumePDF(FPDF):
    def __init__(self, template="modern"):
        super().__init__()
        self.template = template

    def get_colors(self):
        if self.template == "modern":
            return {"primary": (0, 102, 204), "text": (40, 40, 40), "header": (0, 0, 0)}
        elif self.template == "professional":
            return {"primary": (0, 0, 0), "text": (0, 0, 0), "header": (0, 0, 0)}
        elif self.template == "creative":
            return {"primary": (128, 0, 128), "text": (60, 60, 60), "header": (128, 0, 128)}
        return {"primary": (0, 102, 204), "text": (40, 40, 40), "header": (0, 0, 0)}

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(128, 128, 128)
        footer = f'Generated: {datetime.now().strftime("%B %d, %Y")}'
        if self.page_no() > 1:
            footer += f' | Page {self.page_no()}'
        self.cell(0, 10, footer, new_x=XPos.RIGHT, new_y=YPos.TOP, align='C')

def generate_ai_content_safe(prompt, model="gemini-2.5-flash-lite", temp=0.7, max_tokens=800):
    global LAST_CALL_TIME

    for attempt in range(3):
        try:
            with GEMINI_LOCK:
                elapsed = time.time() - LAST_CALL_TIME
                if elapsed < MIN_INTERVAL:
                    wait_time = MIN_INTERVAL - elapsed
                    print(f"⏳ Waiting {wait_time:.1f}s to respect rate limits...")
                    time.sleep(wait_time)

                url = f"https://generativelanguage.googleapis.com/v1/models/{model}:generateContent"
                payload = {
                    "contents": [{"parts": [{"text": prompt}]}],
                    "generationConfig": {
                        "temperature": temp,
                        "maxOutputTokens": max_tokens
                    }
                }

                response = requests.post(url, params={"key": API_KEY}, json=payload, timeout=90)

                LAST_CALL_TIME = time.time()

                if response.status_code == 429:
                    if attempt < 2:
                        print(f"⚠️ Rate limit hit, retrying in {5 * (attempt + 1)}s...")
                        time.sleep(5 * (attempt + 1))
                        continue
                    return "⚠️ Rate limit exceeded. Please wait 60 seconds and try again."

                response.raise_for_status()
                data = response.json()
                result = data["candidates"][0]["content"]["parts"][0]["text"].strip()

                return result

        except requests.exceptions.HTTPError as e:
            if attempt == 2:
                return f"❌ API Error: {e.response.status_code}"
            time.sleep(5)
        except Exception as e:
            if attempt == 2:
                return f"❌ Error: {str(e)[:200]}"
            time.sleep(3)

    return "❌ Failed after 3 attempts."

def build_resume(name, email, phone, linkedin, education, skills, experience,
                target_role, template, industry, include_certs, certifications, include_projects, projects):

    print("🔒 Resume generation started (lock active)")

    if not all([name, email, phone, education, skills, experience, target_role]):
        return None, "❌ Please fill all required fields", ""

    if not re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email):
        return None, "❌ Invalid email format", ""

    status = "🚀 Generating AI-enhanced resume content...\n(Takes ~10-15 seconds to respect free-tier limits)\n\n"

    master_prompt = f"""You are a professional resume writer. Generate a complete resume for {name} applying for {target_role} in {industry}.

**CANDIDATE DETAILS:**
Name: {name}
Target Role: {target_role}
Industry: {industry}
Education: {education}
Skills: {skills}
Experience: {experience}

**OUTPUT FORMAT (use EXACTLY these section markers):**

### PROFESSIONAL_SUMMARY
Write 3-4 lines about {name} in third person. Use "they/their" pronouns. Highlight skills and education. DO NOT invent metrics.

### SKILLS_SECTION
Organize skills into 3-5 categories:
Format: **Category Name**: skill1, skill2, skill3
Categories: Technical Skills, Tools & Platforms, Programming Languages, Cloud/Security, Soft Skills
Use ONLY skills from the list above.

### EXPERIENCE_BULLETS
Create 5-7 bullet points:
- Start with action verbs: Led, Developed, Implemented, Built, Optimized
- Use STAR method
- 1-2 lines each
- Include metrics if provided, otherwise describe impact
- DO NOT invent numbers

### RESUME_SCORE
Rate 1-10 with:
1. Overall Score (X/10): justification
2. Three Strengths
3. Three Improvements
4. ATS Compatibility Score

Keep all sections concise and professional."""

    full_response = generate_ai_content_safe(master_prompt, max_tokens=800)

    if full_response.startswith("❌") or full_response.startswith("⚠️"):
        return None, full_response, ""

    try:
        sections = {}
        current_section = None
        section_content = []

        for line in full_response.split('\n'):
            if line.startswith('### PROFESSIONAL_SUMMARY'):
                if current_section:
                    sections[current_section] = '\n'.join(section_content).strip()
                current_section = 'summary'
                section_content = []
            elif line.startswith('### SKILLS_SECTION'):
                if current_section:
                    sections[current_section] = '\n'.join(section_content).strip()
                current_section = 'skills'
                section_content = []
            elif line.startswith('### EXPERIENCE_BULLETS'):
                if current_section:
                    sections[current_section] = '\n'.join(section_content).strip()
                current_section = 'experience'
                section_content = []
            elif line.startswith('### RESUME_SCORE'):
                if current_section:
                    sections[current_section] = '\n'.join(section_content).strip()
                current_section = 'score'
                section_content = []
            elif current_section:
                section_content.append(line)

        if current_section:
            sections[current_section] = '\n'.join(section_content).strip()

        professional_summary = sections.get('summary', f"{name} is a qualified {target_role} professional with expertise in {industry}.")
        enhanced_skills = sections.get('skills', f"**Technical Skills**: {skills}")
        enhanced_experience = sections.get('experience', experience)
        resume_score = sections.get('score', "Score: 8/10 - Well-structured resume")

        status += "✅ Resume content generated successfully\n"
        status += "✅ Sections parsed\n\n"

    except Exception as e:
        return None, f"❌ Error parsing response: {str(e)}", ""

    pdf = ModernResumePDF(template)
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    colors = pdf.get_colors()

    pdf.set_font('Helvetica', 'B', 24)
    pdf.set_text_color(*colors["header"])
    pdf.cell(0, 12, name.upper(), new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')

    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(60, 60, 60)
    contact = " | ".join([email, phone] + ([linkedin] if linkedin else []))
    pdf.cell(0, 6, contact, new_x=XPos.LMARGIN, new_y=YPos.NEXT, align='C')
    pdf.ln(6)

    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(*colors["header"])
    pdf.cell(0, 8, 'PROFESSIONAL SUMMARY', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_draw_color(*colors["primary"])
    pdf.set_line_width(0.5)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(3)
    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*colors["text"])
    pdf.multi_cell(0, 5, professional_summary)
    pdf.ln(4)

    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(*colors["header"])
    pdf.cell(0, 8, 'EDUCATION', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(3)
    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*colors["text"])
    pdf.multi_cell(0, 5, education)
    pdf.ln(4)

    if include_certs and certifications:
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(*colors["header"])
        pdf.cell(0, 8, 'CERTIFICATIONS', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        pdf.set_font('Helvetica', '', 10)
        pdf.set_text_color(*colors["text"])
        pdf.multi_cell(0, 5, certifications)
        pdf.ln(4)

    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(*colors["header"])
    pdf.cell(0, 8, 'TECHNICAL SKILLS', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(3)
    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*colors["text"])
    pdf.multi_cell(0, 5, enhanced_skills)
    pdf.ln(4)

    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(*colors["header"])
    pdf.cell(0, 8, 'PROFESSIONAL EXPERIENCE', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(3)
    pdf.set_font('Helvetica', '', 10)
    pdf.set_text_color(*colors["text"])
    pdf.multi_cell(0, 5, enhanced_experience)
    pdf.ln(4)

    if include_projects and projects:
        pdf.set_font('Helvetica', 'B', 13)
        pdf.set_text_color(*colors["header"])
        pdf.cell(0, 8, 'KEY PROJECTS', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        pdf.set_font('Helvetica', '', 10)
        pdf.set_text_color(*colors["text"])
        pdf.multi_cell(0, 5, projects)

    unique_id = uuid.uuid4().hex[:8]
    filename = f"{name.replace(' ', '_')}_{unique_id}_Resume.pdf"

    try:
        pdf.output(filename)
    except Exception as e:
        return None, f"❌ PDF generation failed: {str(e)}", ""

    if not os.path.exists(filename):
        return None, "❌ PDF not created", ""

    file_size = os.path.getsize(filename) / 1024
    status += f"📄 PDF: {filename}\n📊 Size: {file_size:.1f} KB\n⚡ Template: {template.title()}"

    print("✅ Resume generation completed successfully")
    return os.path.abspath(filename), status, f"## 📊 Resume Analysis\n\n{resume_score}"

def offensive_malware_explanation(code_snippet, language):
    print("🔒 Malware analysis started (lock active)")
    prompt = f"""⚠️ DEFENSIVE ONLY
Analyze this {language} code for training.

Provide: Purpose, Technique, Impact, Detection, IOCs, Mitigation

Code: {code_snippet[:400]}"""
    result = generate_ai_content_safe(prompt, max_tokens=600)
    return f"## 🔴 Analysis\n\n{result}"

def defensive_malware_plain_english(technical_description):
    print("🔒 Translation started (lock active)")
    prompt = f"""Translate to plain English for non-technical staff.
Use analogies, explain impact.

Description: {technical_description[:400]}"""
    result = generate_ai_content_safe(prompt, max_tokens=500)
    return f"## 🛡️ Plain English\n\n{result}"

def defensive_incident_report(incident_details, severity):
    print("🔒 Incident report generation started (lock active)")
    prompt = f"""Create SOC report. Severity: {severity}

Include: Summary, Timeline, Details, Systems, Actions, Root Cause, Recommendations

Details: {incident_details[:400]}"""
    result = generate_ai_content_safe(prompt, max_tokens=700)
    return f"## 📋 Report\n\n{result}"

def defensive_security_awareness(topic, target_audience):
    print("🔒 Security awareness generation started (lock active)")
    prompt = f"""Security awareness for {target_audience}.
Topic: {topic}

Include: Title, Why It Matters, Threat, Detection, Actions, Tips, Example"""
    result = generate_ai_content_safe(prompt, max_tokens=600)
    return f"## 🎓 Content\n\n{result}"

with gr.Blocks(title="AI Activity Platform") as demo:
    gr.Markdown("# 🤖 AI-Powered Activity Platform\n### Powered by Gemini 2.5 Flash Lite (Free Tier Optimized)")

    with gr.Tabs():
        with gr.Tab("📄 Resume Builder"):
            gr.Markdown("## AI Resume Builder\n**⏱️ Takes ~10-15 seconds (respects free-tier rate limits)**")

            with gr.Row():
                with gr.Column(scale=2):
                    gr.Markdown("### 👤 Personal Information")
                    name_input = gr.Textbox(label="Full Name *", placeholder="Shady Nights")
                    email_input = gr.Textbox(label="Email *", placeholder="Shadynights@gmail.com")
                    phone_input = gr.Textbox(label="Phone *", placeholder="+91 9876543210")
                    linkedin_input = gr.Textbox(label="LinkedIn", placeholder="linkedin.com/in/kashifansari")

                    gr.Markdown("### 🎓 Education")
                    education_input = gr.Textbox(
                        label="Education *",
                        placeholder="B.Tech CSE Cyber Security \nGH Raisoni College, Nagpur\n2024-2028 | CGPA: 8.5",
                        lines=3
                    )

                    gr.Markdown("### 💼 Professional")
                    target_role_input = gr.Textbox(label="Target Role *", placeholder="Cybersecurity Analyst")
                    industry_input = gr.Dropdown(
                        choices=["Technology", "Finance", "Healthcare", "E-commerce", "Cybersecurity", "Education", "Other"],
                        label="Industry *", value="Cybersecurity"
                    )
                    skills_input = gr.Textbox(
                        label="Skills *",
                        placeholder="Python, JavaScript, React.js, Node.js",
                        lines=3
                    )
                    experience_input = gr.Textbox(
                        label="Experience *",
                        placeholder="Cybersecurity Intern | Deloitte | 202\n- Monitored security events, identified 50+ incidents\n- Conducted vulnerability assessments\n- Python automation scripts",
                        lines=8
                    )

                with gr.Column(scale=1):
                    gr.Markdown("### ⚙️ Settings")
                    template_choice = gr.Radio(
                        choices=["modern", "professional", "creative"],
                        label="Template", value="modern"
                    )

                    gr.Markdown("### 📑 Optional")
                    include_certs = gr.Checkbox(label="Certifications", value=False)
                    certifications_input = gr.Textbox(label="Certifications", lines=2, visible=False)
                    include_projects = gr.Checkbox(label="Projects", value=False)
                    projects_input = gr.Textbox(label="Projects", lines=3, visible=False)

                    include_certs.change(lambda x: gr.update(visible=x), [include_certs], [certifications_input])
                    include_projects.change(lambda x: gr.update(visible=x), [include_projects], [projects_input])

                    generate_btn = gr.Button(
                        "🚀 Generate Resume",
                        variant="primary",
                        size="lg",
                        interactive=True
                    )

            with gr.Row():
                with gr.Column():
                    status_output = gr.Textbox(label="Status", lines=6)
                    pdf_output = gr.File(label="📥 Download PDF")
                with gr.Column():
                    score_output = gr.Markdown(label="📊 Analysis")

            generate_btn.click(
                build_resume,
                [name_input, email_input, phone_input, linkedin_input, education_input, skills_input,
                 experience_input, target_role_input, template_choice, industry_input, include_certs,
                 certifications_input, include_projects, projects_input],
                [pdf_output, status_output, score_output]
            )

        with gr.Tab("🔐 Cybersecurity Tools"):
            gr.Markdown("## AI Security Tools")

            with gr.Tabs():
                with gr.Tab("🔴 Malware Analysis"):
                    with gr.Row():
                        with gr.Column():
                            code_input = gr.Textbox(label="Code", lines=10)
                            language_input = gr.Dropdown(
                                choices=["Python", "C", "JavaScript", "PowerShell", "Bash"],
                                label="Language", value="Python"
                            )
                            analyze_btn = gr.Button("🔍 Analyze", variant="primary", interactive=True)
                        with gr.Column():
                            offensive_output = gr.Markdown(label="Results")
                    analyze_btn.click(offensive_malware_explanation, [code_input, language_input], [offensive_output])

                with gr.Tab("🛡️ Plain English"):
                    with gr.Row():
                        with gr.Column():
                            technical_input = gr.Textbox(label="Technical Description", lines=8)
                            translate_btn = gr.Button("🔄 Translate", variant="primary", interactive=True)
                        with gr.Column():
                            plain_output = gr.Markdown(label="Translation")
                    translate_btn.click(defensive_malware_plain_english, [technical_input], [plain_output])

                with gr.Tab("🛡️ Incident Report"):
                    with gr.Row():
                        with gr.Column():
                            incident_input = gr.Textbox(label="Incident", lines=8)
                            severity_input = gr.Radio(choices=["Critical", "High", "Medium", "Low"], label="Severity", value="High")
                            report_btn = gr.Button("📋 Generate", variant="primary", interactive=True)
                        with gr.Column():
                            report_output = gr.Markdown(label="Report")
                    report_btn.click(defensive_incident_report, [incident_input, severity_input], [report_output])

                with gr.Tab("🛡️ Security Awareness"):
                    with gr.Row():
                        with gr.Column():
                            topic_input = gr.Textbox(label="Topic", lines=3)
                            audience_input = gr.Dropdown(
                                choices=["All Employees", "IT Staff", "Executives"],
                                label="Audience", value="All Employees"
                            )
                            awareness_btn = gr.Button("🎓 Generate", variant="primary", interactive=True)
                        with gr.Column():
                            awareness_output = gr.Markdown(label="Content")
                    awareness_btn.click(defensive_security_awareness, [topic_input, audience_input], [awareness_output])

    gr.Markdown("---\n**⚡ Free-Tier Optimized** | Gemini 2.5 Flash Lite | 1 API call per resume | No rate limits")

print("🚀 Launching with FREE-TIER optimizations...")
print("✅ Model: gemini-2.5-flash-lite")
print("✅ Max tokens: 800")
print("✅ Min interval: 10 seconds")
print("✅ Global lock: Active")
demo.launch(share=True, debug=False)


✅ Using API Key: PLACE YOU ... KEY
🚀 Launching with FREE-TIER optimizations...
✅ Model: gemini-2.5-flash-lite
✅ Max tokens: 800
✅ Min interval: 10 seconds
✅ Global lock: Active
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://775c31670c5310fb2d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
